In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from Bio import SeqIO
import os
from collections import Counter

import statsmodels.stats as sm

from matplotlib_venn import venn2
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


from scipy.stats import mannwhitneyu

plt.style.use('tableau-colorblind10')
plt.rcParams.update({'font.family':'Arial'})


In [3]:
base_dir = '/Users/cdubin/VMGC_cervical_dysplasia_paper/code/'

In [4]:
species = pd.read_csv('../../select_species_for_analysis/shared_species_for_analysis.csv')
species = species.set_index('species_id_VMGC')

species_list = species.index.tolist()
species

,species,species_id_GTDB,num_samples_GTDB,num_samples_VMGC
species_id_VMGC,,,,
988598,Lactobacillus crispatus,100122,135,135
240891,Lactobacillus iners,100505,120,121
783244,Bifidobacterium vaginale,100323,78,78
619501,Fannyhessea vaginae,103895,65,43
571325,Lactobacillus jensenii,100515,35,35
611554,Lactobacillus gasseri,100460,20,20


In [5]:
#reordering to group Lactobacillus species
species_list = [988598, 240891, 571325, 611554, 783244, 619501]

In [6]:
[species.loc[i]['species'] for i in species_list]

['Lactobacillus crispatus',
 'Lactobacillus iners',
 'Lactobacillus jensenii',
 'Lactobacillus gasseri',
 'Bifidobacterium vaginale',
 'Fannyhessea vaginae']

In [7]:
combined_genomes = pd.read_csv('../combined_db_build/combined_genomes_with_database.tsv', sep='\t', index_col=0)

In [8]:
combined_genomes

,species,representative,genome_is_representative,fasta_path,database
genome,,,,,
GCF_000439915.2,611554,GCF_000439915.2,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,shared
SRR17284223.mbin.1,240891,SRR17284223.mbin.1,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,VMGC
SRR10258542.mbin.1,240891,SRR17284223.mbin.1,0,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,VMGC
MG329.mbin.1,240891,SRR17284223.mbin.1,0,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,VMGC
P10709414.mbin.1,240891,SRR17284223.mbin.1,0,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,VMGC
...,...,...,...,...,...
GCF_011714655.1,571325,GCF_027155795.1,0,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...,GTDB
GCF_012029675.1,571325,GCF_027155795.1,0,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...,GTDB
GCF_012029775.1,571325,GCF_027155795.1,0,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...,GTDB


In [9]:
genome_to_db_dict = {}

for sp in combined_genomes['species'].unique():

    genome_to_db_dict[sp] = combined_genomes[combined_genomes['species'] == sp]['database'].to_dict()

### Classify centroids as mixed (having both VMGC and GTDB genes) or exclusive to either database

In [10]:
def get_source_db(gene_list, sp):

    dbs = []
    for g in gene_list:

        genome = g.rsplit('_', 1)[0]

        if genome not in genome_to_db_dict[sp]:
            # print(f'{genome} not found')
            continue
        dbs += [genome_to_db_dict[sp][genome]]

    return dbs

def classify_mixing(db_list):


    if len(db_list) == 0:
        return
    db_set = set(db_list)

    if 'shared' in db_set:
        return 'Mixed'
    
    if 'VMGC' in db_set and 'GTDB' in db_set:
        return 'Mixed'
    
    else:
        return db_list[0]


In [11]:
!mkdir centroid_classification

mkdir: centroid_classification: File exists


In [12]:
c90_by_species = {}
for sp in species_list:


    genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')
    c90_genes = genes_info.groupby('centroid_90')['gene_id'].apply(list)

    c90_classified = c90_genes.apply(lambda x: get_source_db(x, sp)).apply(classify_mixing)
    c90_classified.to_frame(name='classification').to_csv(f'centroid_classification/{sp}_C90_classified.csv')
    c90_by_species[sp] = c90_classified
    print(c90_classified.value_counts())
    print()

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/2435058149.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
Mixed    6732
VMGC     4012
GTDB     2686
Name: count, dtype: int64



/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/2435058149.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
VMGC     4558
Mixed    2049
GTDB        2
Name: count, dtype: int64



/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/2435058149.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
Mixed    2064
VMGC     1295
GTDB       67
Name: count, dtype: int64

gene_id
Mixed    3039
VMGC     2045
GTDB      443
Name: count, dtype: int64

gene_id
VMGC     4096
Mixed    2545
GTDB       68
Name: count, dtype: int64

gene_id
VMGC     4744
Mixed    1512
Name: count, dtype: int64



In [13]:
c75_by_species = {}
for sp in species_list:

    genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')
    c75_genes = genes_info.groupby('centroid_75')['gene_id'].apply(list)

    c75_classified = c75_genes.apply(lambda x: get_source_db(x, sp)).apply(classify_mixing)
    c75_classified.to_frame(name='classification').to_csv(f'centroid_classification/{sp}_C75_classified.csv')
    c75_by_species[sp] = c75_classified
    print(c75_classified.value_counts())
    print()

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/1753115465.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
Mixed    5718
VMGC     3144
GTDB     2083
Name: count, dtype: int64



/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/1753115465.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
VMGC     3672
Mixed    1872
Name: count, dtype: int64



/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_15813/1753115465.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  genes_info = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/combined_db/VMGC_GTDB_combined_db/pangenomes/{sp}/genes_info.tsv', sep='\t')


gene_id
Mixed    1927
VMGC     1078
GTDB       56
Name: count, dtype: int64

gene_id
Mixed    2716
VMGC     1593
GTDB      311
Name: count, dtype: int64

gene_id
VMGC     2183
Mixed    2108
GTDB       27
Name: count, dtype: int64

gene_id
VMGC     3065
Mixed    1389
Name: count, dtype: int64

